In [4]:
from entsoe import EntsoePandasClient
from pathlib import Path
import pandas as pd

TOKEN = "4896cc0f-37da-49ba-a3f1-387906966051"
client = EntsoePandasClient(api_key=TOKEN)

# zones to update
ZONES = {
    'DK1': 'DK_1',
    'NO2': 'NO_2',
    'ES':  'ES',
    'DE_LU': 'DE_LU',
    'FR': 'FR'
}

# from where the existing data ends to today
start = pd.Timestamp('20220101', tz='Europe/Brussels')
end   = pd.Timestamp.now(tz='Europe/Brussels')

for zone_name, zone_code in ZONES.items():
    print(f"Fetching {zone_name}...")
    prices = client.query_day_ahead_prices(zone_code, start=start, end=end)
    
    # convert to dataframe and reshape to match previous group's format
    df = prices.to_frame(name='price')
    df.index = df.index.tz_convert('Europe/Vienna')
    df = df.resample('h').mean()
    df = df[~df.index.duplicated(keep='first')]
    df['date'] = df.index.date
    df['hour'] = df.index.hour
    df = df[~df.duplicated(subset=['date', 'hour'], keep='first')]
    df_wide = df.pivot(index='date', columns='hour', values='price')    
    df_wide.columns = [f'h{str(h).zfill(2)}' for h in df_wide.columns]
    
    # load existing preprocessed CSV and append
    existing_path = f'data/clean/{zone_name}_preprocessed.csv'
    
    if Path(existing_path).exists():
        df_existing = pd.read_csv(existing_path, index_col=0)
        df_combined = pd.concat([df_existing, df_wide])
        df_combined = df_combined[~df_combined.index.duplicated(keep='last')]
    else:
        # new zone, no existing file
        df_combined = df_wide
    
    df_combined.to_csv(existing_path)
    print(f"  Saved {existing_path}")

print("Done.")

Fetching DK1...
  Saved data/clean/DK1_preprocessed.csv
Fetching NO2...
  Saved data/clean/NO2_preprocessed.csv
Fetching ES...
  Saved data/clean/ES_preprocessed.csv
Fetching DE_LU...
  Saved data/clean/DE_LU_preprocessed.csv
Fetching FR...
  Saved data/clean/FR_preprocessed.csv
Done.
